# Queen Editor — Colab Host

**Input:** Private GitHub repo `AltanBaysal/Internal-tools` (dal: `prod`) | **Output:** Colab'da derlenmiş frontend (`dist/`)

Repoyu Colab'a çeker ve frontend'i derler. GitHub token'ı **Colab Secrets**'tan okunur — notebook'a yazılmaz.

Sıra:
1. **CONFIG + Clone** — Secrets'tan token al, `prod` dalını çek
2. **Build** — Node ile frontend'i derle (`npm ci && npm run build`)
3. **Hosting** — sonraki adım (tünel + backend)

## 1) CONFIG + Clone — private repoyu `prod` dalından çek

**Tek hazırlık:** Sol menü 🔑 **Secrets** → yeni secret ekle:
- **Name:** `GITHUB_TOKEN`
- **Value:** GitHub fine-grained token (sadece `Internal-tools` reposu, **Contents: Read**)
- **Notebook access:** açık

Sonra bu hücreyi çalıştır. Token notebook'a yazılmaz, sadece Secrets'tan okunur.

In [ ]:
# === GitHub private repo clone (token from Colab Secrets — never hardcoded) ===
import os, shutil, subprocess
from google.colab import userdata

REPO   = "AltanBaysal/Internal-tools"
BRANCH = "prod"
DEST   = "/content/Internal-tools"

# Fine-grained PAT with Contents:Read on this repo. Add it via 🔑 Secrets as GITHUB_TOKEN.
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "❌ GITHUB_TOKEN Secrets'ta yok — sol menü 🔑 Secrets'tan ekle (Contents: Read)"

# Fresh clone each run (idempotent): drop any previous copy.
if os.path.exists(DEST):
    shutil.rmtree(DEST)

url = f"https://{TOKEN}@github.com/{REPO}.git"
print(f"⏳ Klonlanıyor: {REPO} (dal: {BRANCH})")
r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", BRANCH, "--single-branch", url, DEST],
    capture_output=True, text=True,
)
if r.returncode != 0:
    # Mask the token if it ever appears in error output — don't leak the secret.
    raise RuntimeError("❌ Clone başarısız:\n" + (r.stderr or r.stdout).replace(TOKEN, "***"))

print(f"✅ Repo çekildi: {DEST}")
print("📂 queen-editor/:")
for p in sorted(os.listdir(f"{DEST}/queen-editor")):
    print("   ", p)

## 2) Build — frontend'i derle

Colab'ın Node'u ile `npm ci` (commit'li `package-lock.json` kullanılır) + `npm run build`.
Çıktı `queen-editor/frontend/dist/`'e yazılır.

In [ ]:
# === Node + frontend build ===
import os, subprocess

FRONTEND = f"{DEST}/queen-editor/frontend"

def run(cmd, cwd=None, timeout=600):
    """Run a command; non-zero exit -> RuntimeError with the tail of its output (fail-loud)."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f"❌ {' '.join(cmd)}:\n" + (r.stderr or r.stdout)[-1500:])
    return r.stdout

# Colab ships Node 18+; just report the versions.
print("node", run(["node", "-v"]).strip(), "| npm", run(["npm", "-v"]).strip())

print("⏳ npm ci...")
run(["npm", "ci"], cwd=FRONTEND)
print("⏳ npm run build...")
run(["npm", "run", "build"], cwd=FRONTEND)

dist = f"{FRONTEND}/dist"
assert os.path.exists(f"{dist}/index.html"), "❌ Build çıktısı yok (dist/index.html)"
print(f"✅ Build hazır: {dist}")
print("   ", sorted(os.listdir(dist)))

## 3) Hosting — sonraki adım

Build hazır. Bir sonraki aşamada:
- `dist/` cloudflared tüneliyle sunulup tarayıcıdan açılacak
- Gerçek üretim için **FastAPI + ComfyUI köprüsü** eklenecek (frontend'i de FastAPI sunar, `Üret` → photo_generator workflow'u)

Bu notebook şimdilik sadece **çekme + derleme**yi doğruluyor.